1. Chunking (Decoupage)

In [54]:
import json
import tiktoken
from pathlib import Path
from uuid import uuid4

def chunk_text_to_json(
    doc_name: str,
    chunk_size: int = 600,
    overlap_tokens: int = 50,
    encoding_name: str = "o200k_base",
):
    """
    Charge ../data/raw_text/{doc_name}.txt
    Découpe en chunks de `chunk_size` tokens (avec overlap)
    Sauvegarde ../data/chunks/{doc_name}.json

    Chaque chunk possède un unique_id stable.
    """
    raw_text_path = Path("../data/raw_text") / f"{doc_name}.txt"
    if not raw_text_path.exists():
        raise FileNotFoundError(f"Fichier texte introuvable : {raw_text_path}")

    text = raw_text_path.read_text(encoding="utf-8")

    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(text)

    output_dir = Path("../data/chunks")
    output_dir.mkdir(parents=True, exist_ok=True)

    chunks_data = []
    start = 0
    chunk_id = 0
    total_tokens = len(tokens)

    if overlap_tokens >= chunk_size:
        raise ValueError("overlap_tokens doit être < chunk_size")

    while start < total_tokens:
        end = min(start + chunk_size, total_tokens)
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)

        unique_id = str(uuid4())

        chunks_data.append({
            "id": unique_id,
            "content": chunk_text,
            "metadata": {
                "chunk_id": chunk_id,
                "doc_name": doc_name,
                "chunk_size_tokens": len(chunk_tokens),
            }
        })

        chunk_id += 1
        if end == total_tokens:
            break
        start = end - overlap_tokens

    output_path = output_dir / f"{doc_name}.json"
    output_path.write_text(
        json.dumps(chunks_data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    return chunks_data


In [55]:
chunks = chunk_text_to_json(
    doc_name="GDO-Operations-Presence-Electricite",
    chunk_size=50,
    overlap_tokens=10,
)

print(f"\n📦 {len(chunks)} chunks générés")

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk["content"])




📦 1265 chunks générés

--- Chunk 1 ---
 

Direction générale 
de la sécurité civile 
et de la gestion des crises 

GUIDE DE DOCTRINE OPÉRATIONNELLE 

Opérations de secours 
en présence d’électricité  

DSP/SDDRH/BDFE/ JANVIER

--- Chunk 2 ---
SDDRH/BDFE/ JANVIER 2024 
1ère édition  

Ce guide de doctrine opérationnelle a été réalisé en 2023 sous la direction de Eric HOLZMANN 
du bureau en charge de la doctrine, de la

--- Chunk 3 ---
du bureau en charge de la doctrine, de la formation et des équipements, avec l’aide des 
contributeurs suivants :  

Melissa AKLI-CARDIN (SDIS 42), Thierry ARNOULD (SDIS 74), Jéré


In [56]:
def chunk_all_documents():
    """
    Applique le chunking à tous les fichiers .txt présents dans ../data/raw_text/
    en utilisant directement chunk_text_to_json(doc_name).
    """
    raw_text_dir = Path("../data/raw_text")
    txt_files = sorted(raw_text_dir.glob("*.txt"))
    results = {}

    print(f"🚀 Démarrage du chunking batch ({len(txt_files)} documents)")

    for i, txt_path in enumerate(txt_files, start=1):
        doc_name = txt_path.stem
        print(f"📄 [{i}/{len(txt_files)}] Chunking : {doc_name}")

        try:
            chunks = chunk_text_to_json(doc_name=doc_name)
            results[doc_name] = {
                "status": "ok",
                "n_chunks": len(chunks),
            }

        except Exception as e:
            print(f"❌ Erreur pour {doc_name} : {e}")
            results[doc_name] = {
                "status": "error",
                "error": str(e),
            }

    print("\n✅ Chunking batch terminé")


In [57]:
chunk_all_documents()

🚀 Démarrage du chunking batch (13 documents)
📄 [1/13] Chunking : GDO-DRONES-AERIENS-VR29
📄 [2/13] Chunking : GDO-IMPM-V2
📄 [3/13] Chunking : GDO-Interventions-en-milieu-agricole-2019-V2
📄 [4/13] Chunking : GDO-interventions-silos-VF-09-2019
📄 [5/13] Chunking : GDO-Interventions_sur_des_aeronefs_type_ULM_2017
📄 [6/13] Chunking : GDO-Operations-Presence-Electricite
📄 [7/13] Chunking : GDO-PPV_Interventions_elements_photovoltaiques_2017
📄 [8/13] Chunking : GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC
📄 [9/13] Chunking : GDO_Engagement-des-equipes-CYNO_V2
📄 [10/13] Chunking : GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2
📄 [11/13] Chunking : GDO_interventions_a_bord_bateaux_en_eaux_interieures
📄 [12/13] Chunking : GDO_Interventions_dans_les_eoliennes_2019
📄 [13/13] Chunking : GDO_SSUAP_V1.1

✅ Chunking batch terminé


2. Embedding (Vectorisation)

In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
LLM_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def embed_content(chunk: dict, model: str = "text-embedding-3-small") -> list[float]:
    """
    Prend un chunk au format {"content": "...", "metadata": {...}}
    et retourne son embedding (liste de floats).
    """
    text = chunk["content"]

    resp = LLM_client.embeddings.create(
        model=model,
        input=text
    )

    vector = resp.data[0].embedding
    return vector

In [59]:
chunk = chunks[0]                 # 1er chunk
vector = embed_content(chunk)          # vectorisation
print(f"✅ Embedding créé : dimension = {len(vector)}")
print(vector[:10])                   # aperçu

✅ Embedding créé : dimension = 1536
[-0.0007624906720593572, 0.022334638983011246, 0.03847409039735794, -0.05260993540287018, -0.022838613018393517, 0.006490197964012623, -0.005374695174396038, 0.02372363954782486, -0.031934723258018494, -0.015610892325639725]


In [ ]:
def build_vectors_json_for_1_doc_with_batch(
    doc_name: str,
    embedding_model: str = "text-embedding-3-small",
    batch_size: int = 32,
):
    """
    Lit ../data/chunks/{doc_name}.json
    Génère les embeddings OpenAI en batch
    Sauvegarde ../data/vectors/{doc_name}.json
    """

    chunks_path = Path("../data/chunks") / f"{doc_name}.json"
    if not chunks_path.exists():
        raise FileNotFoundError(f"Chunks introuvables : {chunks_path}")

    chunks = json.loads(chunks_path.read_text(encoding="utf-8"))

    output_dir = Path("../data/vectors")
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{doc_name}.json"

    points = []
    total = len(chunks)

    print(f"🚀 Embedding batch → {total} chunks (batch={batch_size})")

    for i in range(0, total, batch_size):
        batch_chunks = chunks[i:i + batch_size]
        texts = [c["content"] for c in batch_chunks]

        # 🔥 UN SEUL CALL pour le batch
        resp = LLM_client.embeddings.create(
            model=embedding_model,
            input=texts
        )

        embeddings = [d.embedding for d in resp.data]

        for chunk, vector in zip(batch_chunks, embeddings):
            chunk_id = chunk["metadata"]["chunk_id"]

            points.append({
                "id": chunk["id"],
                "vector": vector,
                "payload": {
                    "doc_name": doc_name,
                    "chunk_id": chunk_id,
                    "content": chunk["content"],
                }
            })

        print(f"✅ {min(i + batch_size, total)}/{total} embeddings générés")

    out_path.write_text(
        json.dumps(points, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"🎉 Vecteurs sauvegardés → {out_path}")
    return points


In [61]:
build_vectors_json_for_1_doc_with_batch(doc_name="GDO-Operations-Presence-Electricite")

🚀 Embedding batch → 92 chunks (batch=32)
✅ 32/92 embeddings générés
✅ 64/92 embeddings générés
✅ 92/92 embeddings générés
🎉 Vecteurs sauvegardés → ..\data\vectors\GDO-Operations-Presence-Electricite.json


[{'id': '64b197e9-24ee-4421-a9bd-368132afa704',
  'vector': [0.0177141185849905,
   0.020488765090703964,
   0.003695489838719368,
   -0.07405519485473633,
   0.006252041552215815,
   -0.015945129096508026,
   -0.03133290633559227,
   0.03584019094705582,
   -0.028376514092087746,
   -0.015557406470179558,
   0.028885401785373688,
   -0.06741542369127274,
   0.018562262877821922,
   -0.02575938031077385,
   0.061744969338178635,
   0.017592955380678177,
   0.02764953300356865,
   -0.034798186272382736,
   0.02253642864525318,
   -0.010056578554213047,
   0.006091499701142311,
   -0.0316479317843914,
   0.011928556486964226,
   0.04785962030291557,
   -0.040856365114450455,
   0.0004089271533302963,
   -0.0014615359250456095,
   0.032156817615032196,
   0.015060635283589363,
   0.011346970684826374,
   0.041801441460847855,
   -0.031163277104496956,
   -0.052197277545928955,
   -0.02462044358253479,
   0.02263336069881916,
   0.018138190731406212,
   0.0086207902058959,
   0.02503239922

In [62]:
def build_vectors_json_for_all_docs_with_batch():
    """
    Génère les fichiers ../data/vectors/{doc_name}.json pour tous les documents
    présents dans ../data/chunks/ en utilisant build_vectors_json_for_1_doc_with_batch().
    """
    chunks_dir = Path("../data/chunks")
    if not chunks_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {chunks_dir}")

    chunk_files = sorted(chunks_dir.glob("*.json"))
    if not chunk_files:
        print("⚠️ Aucun fichier .json trouvé dans ../data/chunks/")
        return {}

    results = {}
    total_docs = len(chunk_files)

    print(f"🚀 Génération embeddings pour {total_docs} documents ")

    for i, path in enumerate(chunk_files, start=1):
        doc_name = path.stem
        print(f"\n📄 [{i}/{total_docs}] {doc_name}")

        try:
            points = build_vectors_json_for_1_doc_with_batch(
                doc_name=doc_name
            )
            results[doc_name] = {"status": "ok", "n_vectors": len(points)}
            print(f"✅ OK → {len(points)} vecteurs")

        except Exception as e:
            results[doc_name] = {"status": "error", "error": str(e)}
            print(f"❌ Erreur → {e}")

    print("\n🎉 Terminé")


In [63]:
build_vectors_json_for_all_docs_with_batch()

🚀 Génération embeddings pour 13 documents 

📄 [1/13] GDO-DRONES-AERIENS-VR29
🚀 Embedding batch → 2 chunks (batch=32)
✅ 2/2 embeddings générés
🎉 Vecteurs sauvegardés → ..\data\vectors\GDO-DRONES-AERIENS-VR29.json
✅ OK → 2 vecteurs

📄 [2/13] GDO-IMPM-V2
🚀 Embedding batch → 2 chunks (batch=32)
✅ 2/2 embeddings générés
🎉 Vecteurs sauvegardés → ..\data\vectors\GDO-IMPM-V2.json
✅ OK → 2 vecteurs

📄 [3/13] GDO-Interventions-en-milieu-agricole-2019-V2
🚀 Embedding batch → 55 chunks (batch=32)
✅ 32/55 embeddings générés
✅ 55/55 embeddings générés
🎉 Vecteurs sauvegardés → ..\data\vectors\GDO-Interventions-en-milieu-agricole-2019-V2.json
✅ OK → 55 vecteurs

📄 [4/13] GDO-interventions-silos-VF-09-2019
🚀 Embedding batch → 41 chunks (batch=32)
✅ 32/41 embeddings générés
✅ 41/41 embeddings générés
🎉 Vecteurs sauvegardés → ..\data\vectors\GDO-interventions-silos-VF-09-2019.json
✅ OK → 41 vecteurs

📄 [5/13] GDO-Interventions_sur_des_aeronefs_type_ULM_2017
🚀 Embedding batch → 45 chunks (batch=32)
✅ 32/45

3. Indexing (Creation d'une collection QDRANT et chargement des documents dans la collection)

In [9]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models

load_dotenv()

qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

def create_qdrant_collection(
    collection_name: str,
    vector_size: int = 1536,
):
    """
    Crée une collection Qdrant si elle n'existe pas déjà.
    """


    existing_collections = [
        c.name for c in qdrant_client.get_collections().collections
    ]

    if collection_name in existing_collections:
        print(f"ℹ️ Collection déjà existante : {collection_name}")
        return qdrant_client

    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=qdrant_models.VectorParams(
            size=vector_size,
            distance=qdrant_models.Distance.COSINE,
        ),
    )

    print(f"✅ Collection créée : {collection_name}")


In [10]:
COLLECTION_NAME = "rag_noob_collection"
create_qdrant_collection(collection_name=COLLECTION_NAME, vector_size=1536)

ℹ️ Collection déjà existante : rag_noob_collection


In [6]:
collections = qdrant_client.get_collections().collections
print([c.name for c in collections])

NameError: name 'qdrant_client' is not defined

In [ ]:
qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

def index_doc_vectors_to_qdrant(
    doc_name: str = "GDO-Operations-Presence-Electricite",
    batch_size: int = 64,
):
    """
    Charge ../data/vectors/{doc_name}.json (déjà vectorisé)
    et upsert dans Qdrant dans COLLECTION_NAME (déjà créée).
    
    Format attendu dans vectors/{doc_name}.json :
    [
      {"id": "...", "vector": [...], "payload": {...}},
      ...
    ]
    """

    vectors_path = Path("../data/vectors") / f"{doc_name}.json"
    if not vectors_path.exists():
        raise FileNotFoundError(f"Fichier vecteurs introuvable : {vectors_path}")

    items = json.loads(vectors_path.read_text(encoding="utf-8"))
    total = len(items)

    print(f"📦 {doc_name} → {total} vecteurs à indexer dans '{COLLECTION_NAME}'")

    points_buffer = []
    upserted = 0

    for item in items:
        point_id = item["id"]
        vector = item["vector"]
        payload = item.get("payload", {})

        points_buffer.append(
            qdrant_models.PointStruct(
                id=point_id,
                vector=vector,
                payload=payload,
            )
        )

        # flush batch
        if len(points_buffer) >= batch_size:
            qdrant_client.upsert(
                collection_name=COLLECTION_NAME,
                points=points_buffer
            )
            upserted += len(points_buffer)
            points_buffer = []
            print(f"✅ upsert {upserted}/{total}")

    # flush restant
    if points_buffer:
        qdrant_client.upsert(
            collection_name=COLLECTION_NAME,
            points=points_buffer
        )
        upserted += len(points_buffer)

    print(f"🎉 Terminé : {upserted}/{total} vecteurs indexés dans '{COLLECTION_NAME}'")
    return upserted


In [68]:
index_doc_vectors_to_qdrant()

📦 GDO-Operations-Presence-Electricite → 92 vecteurs à indexer dans 'rag_noob_collection'
✅ upsert 64/92
🎉 Terminé : 92/92 vecteurs indexés dans 'rag_noob_collection'


92

In [69]:
def index_all_docs_vectors_to_qdrant():
    """
    Indexe tous les fichiers ../data/vectors/{doc_name}.json dans Qdrant
    en réutilisant index_doc_vectors_to_qdrant().
    """
    vectors_dir = Path("../data/vectors")
    if not vectors_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {vectors_dir}")

    vector_files = sorted(vectors_dir.glob("*.json"))
    if not vector_files:
        print("⚠️ Aucun fichier .json trouvé dans ../data/vectors/")
        return {}

    results = {}
    total_docs = len(vector_files)

    print(f"🚀 Indexation Qdrant pour {total_docs} documents")

    for i, vf in enumerate(vector_files, start=1):
        doc_name = vf.stem
        print(f"\n📄 [{i}/{total_docs}] Indexation : {doc_name}")

        try:
            n = index_doc_vectors_to_qdrant(doc_name=doc_name)
            results[doc_name] = {"status": "ok", "n_vectors": n}
        except Exception as e:
            print(f"❌ Erreur pour {doc_name} : {e}")
            results[doc_name] = {"status": "error", "error": str(e)}

    print("\n🎉 Indexation batch terminée")
    return results


In [70]:
index_all_docs_vectors_to_qdrant()

🚀 Indexation Qdrant pour 13 documents

📄 [1/13] Indexation : GDO-DRONES-AERIENS-VR29
📦 GDO-DRONES-AERIENS-VR29 → 2 vecteurs à indexer dans 'rag_noob_collection'
🎉 Terminé : 2/2 vecteurs indexés dans 'rag_noob_collection'

📄 [2/13] Indexation : GDO-IMPM-V2
📦 GDO-IMPM-V2 → 2 vecteurs à indexer dans 'rag_noob_collection'
🎉 Terminé : 2/2 vecteurs indexés dans 'rag_noob_collection'

📄 [3/13] Indexation : GDO-Interventions-en-milieu-agricole-2019-V2
📦 GDO-Interventions-en-milieu-agricole-2019-V2 → 55 vecteurs à indexer dans 'rag_noob_collection'
🎉 Terminé : 55/55 vecteurs indexés dans 'rag_noob_collection'

📄 [4/13] Indexation : GDO-interventions-silos-VF-09-2019
📦 GDO-interventions-silos-VF-09-2019 → 41 vecteurs à indexer dans 'rag_noob_collection'
🎉 Terminé : 41/41 vecteurs indexés dans 'rag_noob_collection'

📄 [5/13] Indexation : GDO-Interventions_sur_des_aeronefs_type_ULM_2017
📦 GDO-Interventions_sur_des_aeronefs_type_ULM_2017 → 45 vecteurs à indexer dans 'rag_noob_collection'
🎉 Terminé 

{'GDO-DRONES-AERIENS-VR29': {'status': 'ok', 'n_vectors': 2},
 'GDO-IMPM-V2': {'status': 'ok', 'n_vectors': 2},
 'GDO-Interventions-en-milieu-agricole-2019-V2': {'status': 'ok',
  'n_vectors': 55},
 'GDO-interventions-silos-VF-09-2019': {'status': 'ok', 'n_vectors': 41},
 'GDO-Interventions_sur_des_aeronefs_type_ULM_2017': {'status': 'ok',
  'n_vectors': 45},
 'GDO-Operations-Presence-Electricite': {'status': 'ok', 'n_vectors': 92},
 'GDO-PPV_Interventions_elements_photovoltaiques_2017': {'status': 'ok',
  'n_vectors': 49},
 'GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC': {'status': 'ok',
  'n_vectors': 34},
 'GDO_Engagement-des-equipes-CYNO_V2': {'status': 'ok', 'n_vectors': 2},
 'GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2': {'status': 'ok',
  'n_vectors': 77},
 'GDO_interventions_a_bord_bateaux_en_eaux_interieures': {'status': 'ok',
  'n_vectors': 46},
 'GDO_Interventions_dans_les_eoliennes_2019': {'status': 'ok',
  'n_vectors': 18},
 'GDO_SSUAP_V1.1': {'statu

4. Retrieving

In [2]:
import os
from qdrant_client import QdrantClient

def search_top_chunks(
    question: str,
    top_k: int = 5,
    query_filter=None,   # optionnel, tu peux laisser None
):
    """
    - vectorise la question via embed_content
    - interroge Qdrant avec query_points
    - retourne les top_k chunks (payload)
    """

    # 1) Client Qdrant
    qdrant_client = QdrantClient(
        url=os.getenv("QDRANT_URL"),
        api_key=os.getenv("QDRANT_API_KEY"),
    )

    # 2) Embedding de la question (réutilise embed_content)
    question_vector = embed_content({"content": question})

    # 3) Recherche Qdrant (API query_points)
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=question_vector,
        limit=top_k,
        with_payload=True,
        query_filter=query_filter,
    ).points

    # 4) Extraction propre
    chunks = []
    for rank, hit in enumerate(hits, start=1):
        payload = hit.payload or {}

        # selon version, le score peut être hit.score ou hit.distance
        score = getattr(hit, "score", None)
        if score is None:
            score = getattr(hit, "distance", None)

        chunks.append({
            "rank": rank,
            "score": score,
            "doc_name": payload.get("doc_name"),
            "chunk_id": payload.get("chunk_id"),
            "content": payload.get("content"),
        })

    return chunks


In [12]:
question = "Hello"
chunks = search_top_chunks(question, top_k=5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")
    print(c["content"][:400])


--- Rank 1 | score=0.3195141 ---
Doc: GDO-Interventions_sur_des_aeronefs_type_ULM_2017 | chunk: 0
 	


	
 

	


!	
	"#!	"
$%&'()'()'*+,-&.()*/0-1,&*..(22()
$34)567896:
;<=>?<@A>@B?@CDEF=GH>@
A>@B?@ID=J?FGDH@
>F@A>K@LM<GN>J>HFK
OHF>=P>HFGDHK@K<=@A>K@?Q=DH>RK@A>@FSN>@TBF=?@UQV>=@WDFD=GKQ

 

 














 


--- Rank 2 | score=0.31773221 ---
Doc: GDO-PPV_Interventions_elements_photovoltaiques_2017 | chunk: 43
)>?
 ,'-./012#((#%30%$4-#31
567"869+ 9 
 6:>&-#$,6"@A!6B"
+6<+CD
 6: ,-#(4%E,19AF
+6<+GH
A6=I,#&#EE19)@" 
+6<+GH
!#$%&'( JA<!:B"><@
<!"+8 "A
"-#$ )K>"@@"
L'44M'N8 )@A+
I,#&#EE1JAB"@@"
L'44M'N8 )@A+
 ,-#(4%E,1@AF
"&1$4-#$'&+'O14NI-%0P$4(
HD

 
		



		
 !""#"$%$#!!&
! $

--- Ra

5. Retrieval Augmented Generation

In [13]:
def ask_llm_with_memory(
    question: str,
    memory: list | None = None
):
    """
    Envoie une question au LLM avec :
    - un system prompt TOUJOURS présent
    - une mémoire user/assistant
    """
    SYSTEM_PROMPT = """
Réponds courtoisement aux questions sociales, ou utilises le contexte fourni pour les questions techniques. 
Tu dois répondre uniquement à partir du contexte fournis. 
Si l'information n'est pas présente dans le contexte, réponds explicitement : "Je ne sais pas".
"""
    
    if memory is None:
        memory = []

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + list(memory)
        + [{"role": "user", "content": question}]
    )

    model_response = LLM_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2,
    )

    answer = model_response.choices[0].message.content

    updated_memory = list(memory) + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]

    return model_response, answer, updated_memory

In [14]:
def ask_question_with_rag(
    question: str,
    top_k: int = 5,
    memory: list | None = None,
):
    """
    Pose une question en mode RAG :
    - recherche les top_k chunks via search_top_chunks
    - construit un contexte à partir de ces chunks
    - interroge le LLM via ask_llm_with_memory
    - retourne (model_response, answer, updated_memory)
    """

    # 1) Recherche des chunks pertinents
    chunks = search_top_chunks(question, top_k=top_k)

    if not chunks:
        raise ValueError("Aucun chunk trouvé pour la question")

    # 2) Construction du contexte
    context_blocks = []
    for c in chunks:
        block = f"""
--- Source ---
Document: {c['doc_name']}
Chunk: {c['chunk_id']}

{c['content']}
""".strip()
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)

    # 3) Prompt final
    full_question = f"""

=== CONTEXTE ===
{context}

=== QUESTION ===
{question}
""".strip()

    # 4) Appel LLM
    model_response, answer, updated_memory = ask_llm_with_memory(
        full_question,
        memory
    )

    return model_response, answer, updated_memory


In [15]:
question = "Quels sont les risques liés à la présence d’électricité lors d’une intervention ?"

model_response, answer, memory = ask_question_with_rag(
    question,
    top_k=5
)

print(answer)

print("\n--- Token Usage ---")
print("Prompt tokens     :", model_response.usage.prompt_tokens)
print("Completion tokens :", model_response.usage.completion_tokens)
print("Total tokens      :", model_response.usage.total_tokens)

Les risques liés à la présence d’électricité lors d’une intervention incluent :

1. **Accidents électriques** : Ces accidents peuvent être causés par le contact avec deux conducteurs sous tension ou un conducteur sous tension et la terre, entraînant un passage de courant dans le corps (contact direct), ou par contact indirect avec une pièce conductrice mise accidentellement sous tension.

2. **Amorçage** : S'approcher d'une ligne électrique sans la toucher peut provoquer un arc électrique, avec un risque d'électrocution.

3. **Chocs électriques** : Ils peuvent survenir lors d'un contact direct avec une partie active d'un conducteur électrique.

4. **Électrisation et électrocution** : L'électrisation désigne les lésions causées par le passage d'un courant électrique à travers le corps, tandis que l'électrocution se réfère à une électrisation mortelle.

5. **Brûlures électriques** : Elles peuvent résulter de la destruction de la peau et des tissus en profondeur.

6. **Traumatismes conton